# EDA 01: Enterprise Sales Overview & Trend Analysis

This notebook analyzes total sales revenue, transaction volumes, growth trajectories over time, and sales distributions across raw dataset sources and geographic regions.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)

print("Loading sales_fact.parquet...")
df_sales = pd.read_parquet('data/processed/sales_fact.parquet')
print(f"Total Rows: {len(df_sales):,}, Columns: {df_sales.shape[1]}")
print(f"Date Range: {df_sales['date'].min()} to {df_sales['date'].max()}")
df_sales.head()


Loading sales_fact.parquet...
Total Rows: 1,143,942, Columns: 14
Date Range: 2010-02-05 00:00:00 to 2018-09-03 09:06:57


In [2]:
# Monthly Aggregation of Total Revenue & Transaction Counts
df_sales['year_month'] = df_sales['date'].dt.to_period('M')
monthly_sales = df_sales.groupby('year_month').agg(
    total_revenue=('total_sales', 'sum'),
    transaction_count=('sales_id', 'count'),
    avg_ticket=('total_sales', 'mean')
).reset_index()
monthly_sales['year_month_str'] = monthly_sales['year_month'].astype(str)

fig, ax1 = plt.subplots(figsize=(14, 6))
ax2 = ax1.twinx()

ax1.plot(monthly_sales['year_month_str'], monthly_sales['total_revenue'] / 1e6, 'b-o', label='Monthly Revenue ($M)', linewidth=2)
ax2.plot(monthly_sales['year_month_str'], monthly_sales['transaction_count'], 'g--s', label='Transaction Count', linewidth=2)

ax1.set_xlabel('Year-Month')
ax1.set_ylabel('Total Revenue ($ Millions)', color='b')
ax2.set_ylabel('Transaction Count', color='g')
plt.title('Enterprise Monthly Sales Revenue and Transaction Volume (2010 - 2018)')
ax1.tick_params(axis='x', rotation=45)
fig.tight_layout()
plt.show()


In [3]:
# Revenue & Volume Breakdown by Dataset Source
source_summary = df_sales.groupby('dataset_source').agg(
    total_revenue=('total_sales', 'sum'),
    total_volume=('quantity', 'sum'),
    transaction_count=('sales_id', 'count'),
    mean_sale=('total_sales', 'mean'),
    median_sale=('total_sales', 'median')
).reset_index()

source_summary['revenue_share_pct'] = (source_summary['total_revenue'] / source_summary['total_revenue'].sum()) * 100
print(source_summary)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=source_summary, x='dataset_source', y='total_revenue', ax=ax1, palette='viridis')
ax1.set_title('Total Gross Revenue by Dataset Source ($)')
ax1.set_ylabel('Revenue ($)')

sns.barplot(data=source_summary, x='dataset_source', y='transaction_count', ax=ax2, palette='magma')
ax2.set_title('Transaction Volume by Dataset Source')
ax2.set_ylabel('Transaction Count')
plt.tight_layout()
plt.show()


  dataset_source  total_revenue  ...    median_sale  revenue_share_pct
0         dataco   3.631322e+07  ...     199.919998           0.287938
1             m5   6.710984e+09  ...  960746.040000          53.213268
2          olist   1.298820e+07  ...      74.990000           0.102987
3       rossmann   5.851200e+09  ...    6369.000000          46.395807

[4 rows x 7 columns]


In [4]:
# Top 10 Geographic Regions by Revenue
region_summary = df_sales.groupby('region').agg(
    total_revenue=('total_sales', 'sum'),
    transaction_count=('sales_id', 'count')
).reset_index().sort_values(by='total_revenue', ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(data=region_summary.head(10), x='total_revenue', y='region', palette='Spectral')
plt.title('Top 10 Regions by Total Revenue ($)')
plt.xlabel('Total Revenue ($)')
plt.ylabel('Region')
plt.tight_layout()
plt.show()


In [5]:
# Statistical Summary of Sales Metrics
sales_stats = df_sales[['total_sales', 'quantity', 'unit_price', 'shipping_cost', 'profit']].describe().T
sales_stats['skewness'] = df_sales[['total_sales', 'quantity', 'unit_price', 'shipping_cost', 'profit']].skew()
print("Sales Fact Descriptive Statistics:")
print(sales_stats)


Sales Fact Descriptive Statistics:
                   count          mean  ...           max   skewness
total_sales    1143942.0  11024.584726  ...  2.404035e+06  17.509434
quantity       1143942.0    563.441950  ...  7.388000e+03   1.473777
unit_price     1143942.0   5930.651757  ...  3.818686e+06  17.892306
shipping_cost  1143942.0      1.968552  ...  4.096800e+02   8.789015
profit         1143942.0      3.467748  ...  9.118000e+02 -10.019216

[5 rows x 9 columns]
